# Extract power doppler images and output to NIfTI
[Also see `fUSI_to_BIDS_session.py` for loading all runs in a script.]

This notebook we extract power doppler images from chestnuts output into a NifTI format using nibabel and nilearn.

It will save three files in the same folder:
1) NIFTI nii.gz containing the PD images
2) JSON for metadata ()
3) probe_events.csv for the pd h5 filename and corresponding timestamps
4) animation for quick overview

### Resources:

**Nilearn quick start** 
https://nilearn.github.io/dev/quickstart.html

**I/O**
https://nilearn.github.io/dev/manipulating_images/input_output.html#extracting-data

**Affine**
https://nipy.org/nibabel/coordinate_systems.html#the-affine-by-example

See more implementation in `SessionLoader.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import anise.utils
from anise.gui import MakeAnimation
from anise.SessionLoader import SessionLoader
from IPython.display import HTML, Video


In [ ]:
#################################################################
#      Define data paths and choose output path location        #
#################################################################

save_output_on_local = False
root = Path.home() /'cassini/UCLA_collaboration/'
base_path = root / '2024-06-07/UCLA_006/'
run_id = 15
acqusition_id = 0 # switch between sequences if there are multiple (default: 0)

# define output path to save data, plots, and videos (default: /fUSI_corrected)
if save_output_on_local:
    output_path = Path.home() / "Downloads" / "UCLA_fUSI_BIDS"  # save to user defined location e.g. (/Downloads folder)
else:
    # default to save in the same root folder
    output_path = root / "UCLA_fUSI_BIDS"

# Find path using sessionLoader
ses = SessionLoader(root, base_path, run_id, acqusition_id, output_path=output_path)

In [ ]:
#################################################################
#                Load power doppler from h5                     #
#################################################################

# process power doppler files based on same number of tissue components in clutter filter (PCA)
unique_num_tissue_components = ses.find_unique_num_tissue_components_within_single_acqusition()

# Parse separately if more than one tissue components within a single acqusition
for n_tc in unique_num_tissue_components:
    try:
        ses.filter_power_doppler_files(num_tissue_components=n_tc)
        
        # load fusi data
        fusi_data = ses.load_fusi_frames()

    except:
        continue

In [ ]:
#################################################################
#         Load task event and add timing offset                 #
#################################################################

# task_name = '' # name can be specified in extract_task_events(), otherwise automatically detected (e.g. audio, light, SSEP)
# task_description = '' # description can be specified in extract_task_events(), otherwise automatically added
ses.extract_task_events()

# calculate offset between task start time and ensemble start time
behavior_offset = (ses.probe_events['global_start_time'][0] - ses.task_start).total_seconds()
print('\n\tpwd ensemble start time:', ses.probe_events['global_start_time'][0])
print('-\ttask start time: \t', ses.task_start)
print('___________________________________________________________')
print(f'Adjusting behavioral offset by \t\t   {behavior_offset} seconds')

# adjust task events onset time
ses.task_events['onset'] = ses.task_events['onset'] - behavior_offset
ses.task_events


In [ ]:
#################################################################
#                          save outputs                         #
#################################################################

fus_dir = ses.output_path / 'sourcedata' / f'sub-{ses.subject_id}' / f'ses-{ses.session_id}' / 'fus'
beh_dir = ses.output_path / 'sourcedata' / f'sub-{ses.subject_id}' / f'ses-{ses.session_id}' / 'beh'

# get sidecar json metadata
ses.get_sidecar_json()

filter = ses.sidecar['ClutterFilters'][0]['FilterType'].lower()
date = ses.session_id
sub = ses.subject_id
task = ses.task_name
run = ses.run
acq = ses.acqusition_id
plane = ses.plane
if ses.plane is not None:
    bmode_base = (f'sub-{sub}_task-{task}_run-{run}_acq-{acq}_pose-{plane}')
    pd_base = (bmode_base + f'_proc-{filter}{n_tc}ntc')
else:          
    bmode_base = (f'sub-{sub}_task-{task}_run-{run}_acq-{acq}')
    pd_base = (bmode_base + f'_proc-{filter}{n_tc}ntc')

# save to NIFTI and metadata output_path
filename = ses.save_to_nifti(fusi_data, fus_dir, pd_base)

# save task events (need to run after save_to_nifti for self.output_filename)
ses.save_task_events(beh_dir, pd_base + '_events.tsv')


In [ ]:
#################################################################
#           Display and save power doppler movie                #
#################################################################

ani = None
file_path = ses.output_path / 'derivatives' / 'registration' / f'sub-{ses.subject_id}' / f'ses-{ses.session_id}'
file_path.mkdir(parents=True, exist_ok=True)
if fusi_data is not None:
    ani = MakeAnimation(fusi_data[:, 0], 
                        output_file=str( file_path / f'{filename}_before.mp4'),  # mp4
                        fps=10, 
                        depth=ses.sidecar['Depth'], 
                        lateral=ses.sidecar['Lateral'], 
                        time=ses.sidecar['VolumeTiming'],
    )

    ani = MakeAnimation(fusi_data[:, 0],
                        output_file=str(file_path / f'{filename}_before.gif'), # gif
                        fps=10, 
                        depth=ses.sidecar['Depth'], 
                        lateral=ses.sidecar['Lateral'], 
                        time=ses.sidecar['VolumeTiming'],
    )
HTML(ani.to_jshtml())